# CertValidator — Phase 1 Dataset Exploration
This notebook walks through:
1. Generating synthetic certificates
2. Running preprocessing
3. Visualising ELA forensics
4. Exploring the dataset class

In [ ]:
import sys; sys.path.insert(0, '..')
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from PIL import Image

plt.rcParams.update({'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
                     'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
                     'xtick.color': '#64748b', 'ytick.color': '#64748b'})

## 1. Generate synthetic certificates

In [ ]:
from scripts.generate_synthetic import CertificateGenerator

gen = CertificateGenerator(seed=42)

# Generate one genuine + one fake
genuine_img, genuine_data = gen.generate_genuine()
fake_img,    fake_data    = gen.generate_fake()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(genuine_img); axes[0].set_title('Genuine certificate', color='#4ade80'); axes[0].axis('off')
axes[1].imshow(fake_img);    axes[1].set_title(f'Fake — tamper: {fake_data.tamper_type}', color='#f87171'); axes[1].axis('off')
plt.tight_layout()
plt.show()

print('Genuine name:', genuine_data.student_name)
print('Fake name:   ', fake_data.student_name)

## 2. Preprocessing pipeline

In [ ]:
from ml.src.preprocessing.pipeline import CertificatePreprocessor
import tempfile, os

pp = CertificatePreprocessor()

# Save genuine cert to temp file and preprocess
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    tmp_path = f.name
genuine_img.save(tmp_path)

result = pp.process(tmp_path)
print('Metadata:', result.metadata)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['Original', 'Preprocessed (BGR→RGB)', 'Grayscale']
imgs   = [
    np.array(genuine_img),
    cv2.cvtColor(result.processed_image, cv2.COLOR_BGR2RGB),
    result.grayscale,
]
cmaps  = [None, None, 'gray']
for ax, img, title, cmap in zip(axes, imgs, titles, cmaps):
    ax.imshow(img, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
os.unlink(tmp_path)

## 3. ELA forensics — the key differentiator

In [ ]:
from ml.src.utils.ela_analysis import compute_ela, ela_heatmap, overlay_ela

# Simulate a tampered certificate by painting a region
tampered = result.processed_image.copy()
cx = tampered.shape[1] // 2
tampered[600:700, cx-300:cx+300] = (255, 255, 200)  # paint-over region

ela_genuine  = compute_ela(result.processed_image)
ela_tampered = compute_ela(tampered)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
rows = [
    (result.processed_image, ela_genuine,  'Genuine', '#4ade80'),
    (tampered,               ela_tampered, 'Tampered (name region painted)', '#f87171'),
]
for row_idx, (img, ela, label, color) in enumerate(rows):
    axes[row_idx, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[row_idx, 0].set_title(f'{label} — original', color=color)
    axes[row_idx, 1].imshow(cv2.cvtColor(ela, cv2.COLOR_BGR2RGB))
    axes[row_idx, 1].set_title(f'{label} — ELA', color=color)
    axes[row_idx, 2].imshow(cv2.cvtColor(overlay_ela(img, ela, alpha=0.5), cv2.COLOR_BGR2RGB))
    axes[row_idx, 2].set_title(f'{label} — heatmap overlay', color=color)
    for ax in axes[row_idx]: ax.axis('off')
plt.suptitle('ELA reveals tampering invisible to the human eye', color='#e2e8f0', fontsize=14)
plt.tight_layout(); plt.show()

# Quantify the difference
region_genuine  = ela_genuine[600:700, cx-300:cx+300].mean()
region_tampered = ela_tampered[600:700, cx-300:cx+300].mean()
print(f'Mean ELA in tampered region — genuine: {region_genuine:.1f} | tampered: {region_tampered:.1f}')
print(f'Ratio: {region_tampered/region_genuine:.1f}× higher — forensic signal confirmed')

## 4. Dataset class smoke test

In [ ]:
DATA_ROOT = '../ml/data/synthetic'

if not Path(DATA_ROOT).exists():
    print('No synthetic data yet. Run:')
    print('  python scripts/generate_synthetic.py --count 200 --output ml/data/synthetic')
else:
    from ml.src.dataset.certificate_dataset import build_dataloaders
    loaders = build_dataloaders(DATA_ROOT, batch_size=4, num_workers=0)
    batch   = next(iter(loaders['train']))
    print('combined tensor shape:', batch['combined'].shape)  # (4, 6, 768, 1088)
    print('labels:', batch['label'].tolist())
    print('tamper types:', batch['tamper_type'])
    
    # Visualise one pair
    img_np = batch['combined'][0, :3].permute(1, 2, 0).numpy()
    ela_np = batch['combined'][0, 3:].permute(1, 2, 0).numpy()
    # Undo ImageNet normalisation for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = np.clip(img_np * std + mean, 0, 1)
    ela_np = np.clip(ela_np * std + mean, 0, 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(img_np); axes[0].set_title('RGB channel'); axes[0].axis('off')
    axes[1].imshow(ela_np); axes[1].set_title('ELA channel'); axes[1].axis('off')
    plt.tight_layout(); plt.show()